## Scenario 2: Handling Large Datasets (Design + Simulation)

This scenario focuses on designing a **robust PySpark ingestion pipeline** for handling a **large daily CSV file (~60GB)** arriving in cloud storage.

Since this notebook does not have access to an actual large file, the goal here is to demonstrate **production-grade ingestion design**, including:

- Explicit schema usage (no inference)
- Efficient ingestion patterns for large files
- Partitioning strategy for scalability
- Small-files avoidance
- Safe restart and retry behavior

The ingestion logic is therefore **simulated using in-memory sample data**, while the **production read path is clearly documented and explained**.

### Design Assumptions & Key Considerations

This ingestion pipeline is designed with the following **production assumptions** and **scalability considerations** in mind.

#### Design Assumptions
- Data arrives daily as a **large CSV file (~60GB)** in cloud storage (e.g., S3 / ADLS / GCS)
- Files are **append-only CDC deltas**, not full snapshots
- Raw ingestion is intentionally **decoupled from downstream de-duplication** (handled in Scenario 1)
- **Delta Lake** is used to provide transactional guarantees and atomic commits

#### Key Design Considerations

1. **Explicit Schema**
   - Avoids expensive schema inference on large files
   - Prevents schema drift and silent type changes

2. **Large File Handling**
   - Leverages Spark's distributed reads
   - Controls partition sizes to avoid executor OOMs

3. **Small-Files Avoidance**
   - Writes data using controlled repartitioning
   - Ensures efficient downstream queries and metadata handling

4. **Failure Recovery & Restart Safety**
   - Job can be safely re-run without duplicating data
   - Delta Lake guarantees that partial writes do not corrupt tables

#### Step 1: Define Explicit Schema (No Inference)

**Why explicit schema?**

- Schema inference on large files is expensive and slow
- Prevents silent schema drift
- Ensures predictable performance and data quality

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    BooleanType,
    DateType,
)

customer_schema = StructType(
    [
        StructField("party_key", StringType(), False),
        StructField("source_updated_at", TimestampType(), False),
        StructField("name", StringType(), True),
        StructField("dob", DateType(), True),
        StructField("country", StringType(), True),
        StructField("is_deleted", BooleanType(), False),
        StructField("ingested_at", TimestampType(), False),
    ]
)

#### Step 2: Efficient Ingestion of Large CSV (Production Pattern)

**Key options explained**
- `schema(...)` → no inference
- `FAILFAST` → stops on corrupt records
- Distributed read → Spark splits file across executors

```python
# Production-style ingestion (documented, not executed here)
input_df = (
    spark.read
    .format("csv")
    .schema(customer_schema)
    .option("header", "true")
    .option("mode", "FAILFAST")
    .load("s3://bucket/raw/customers/2024-01-10/")
)
```

#### Step 3: Simulation Using In-Memory Data

Since we cannot load a real 60GB file in this environment, we simulate ingestion using a small in-memory DataFrame that follows the same schema.

In [0]:
from pyspark.sql import Row
from datetime import datetime, date

sample_data = [
    Row(
        "1",
        datetime(2024, 1, 10, 9, 0),
        "Alice",
        date(1990, 1, 1),
        "US",
        False,
        datetime(2024, 1, 10, 9, 2),
    ),
    Row(
        "2",
        datetime(2024, 1, 10, 8, 30),
        "Bob",
        date(1985, 5, 5),
        "IN",
        False,
        datetime(2024, 1, 10, 8, 35),
    ),
    Row(
        "3",
        datetime(2024, 1, 10, 7, 45),
        "Charlie",
        date(1992, 7, 7),
        "UK",
        False,
        datetime(2024, 1, 10, 7, 50),
    ),
]

simulated_df = spark.createDataFrame(sample_data, schema=customer_schema)
display(simulated_df)

party_key,source_updated_at,name,dob,country,is_deleted,ingested_at
1,2024-01-10T09:00:00.000Z,Alice,1990-01-01,US,false,2024-01-10T09:02:00.000Z
2,2024-01-10T08:30:00.000Z,Bob,1985-05-05,IN,false,2024-01-10T08:35:00.000Z
3,2024-01-10T07:45:00.000Z,Charlie,1992-07-07,UK,false,2024-01-10T07:50:00.000Z


#### Step 4: Partitioning Strategy (Small-Files Avoidance)

For large ingestions, repartitioning before write helps:
- Control file sizes
- Avoid excessive small Delta files
- Improve downstream query performance

In [0]:
optimized_df = simulated_df.repartition(8, "party_key")

#### Step 5: Write to Raw Delta Table (Idempotent & Safe)

In [0]:
(
    optimized_df.write.format("delta")
    .mode("append")
    .saveAsTable("demo_raw_customers_delta_large")
)

### Outcome of Scenario 2

This scenario demonstrates how to **design** a large-scale ingestion pipeline even when working in a constrained environment:

- Explicit schema ensures predictable performance
- Distributed reads scale to very large files
- Controlled repartitioning prevents small-file problems
- Delta Lake guarantees safe retries and atomic writes
- Downstream de-duplication remains isolated (Scenario 1)

Together, this pattern enables **safe, scalable, production-ready ingestion** for very large daily datasets.

## Scenario 2 (Extended): Production Hardening & Reusability

This section extends Scenario 2 with **production-grade enhancements** that are commonly expected in real-world large-scale ingestion pipelines:

1. Performance tuning knobs
2. Failure safety / injection and restart guarantees
3. Reusable ingestion framework patterns

These additions demonstrate how the ingestion pipeline scales safely, fails predictably, and can be reused across datasets.

### Performance Tuning Knobs

- For very large files (50-100GB+), default Spark settings are often insufficient.
- The following knobs help control memory pressure, task sizing, and file layout.

In [0]:
# ------------------------------------------------------------
# Spark Performance Tuning for Large-File Ingestion (Scenario 2)
# ------------------------------------------------------------

# Control how Spark splits large input files into tasks
# Ensures each task processes ~128MB, preventing executor OOMs
spark.conf.set("spark.sql.files.maxPartitionBytes", 128 * 1024 * 1024)  # 128 MB

# Limit the number of shuffle partitions to avoid small files
# This should roughly align with cluster core count
spark.conf.set("spark.sql.shuffle.partitions", 200)

### Delta Lake Optimization Notes

Delta write optimizations such as **optimizeWrite** and **autoCompact** are typically enabled at the **cluster level** in Databricks and are not always settable at runtime (especially in Spark Connect environments).

Therefore, this design relies on:
1. Controlled repartitioning during writes
2. Explicit `OPTIMIZE` commands where supported

In [0]:
# Optional optimization step (Databricks-supported)
spark.sql(
    """
OPTIMIZE demo_raw_customers_delta_large
"""
)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

### Inspecting Delta OPTIMIZE Metrics

The `OPTIMIZE` command in Delta Lake returns a **metrics DataFrame** describing what the optimization actually did.

Instead of treating OPTIMIZE as a black box, we explicitly inspect a small, high-value subset of metrics to validate:

- Small-file compaction effectiveness
- Write amplification
- Execution cost

This is a common production practice to ensure OPTIMIZE jobs are providing value and not rewriting data unnecessarily.

In [0]:
from pyspark.sql import functions as F

# Run OPTIMIZE and capture execution metrics
optimize_metrics = spark.sql(
    """
OPTIMIZE demo_raw_customers_delta_large
"""
)

# Select only the most relevant operational metrics
optimize_metrics.select(
    "path",
    F.col("metrics.numFilesRemoved").alias("files_removed"),
    F.col("metrics.numFilesAdded").alias("files_added"),
    F.col("metrics.filesAdded.totalSize").alias("bytes_written"),
    F.col("metrics.totalFilesSkipped").alias("files_skipped"),
    F.col("metrics.totalTaskExecutionTimeMs").alias("execution_time_ms"),
).display()

path,files_removed,files_added,bytes_written,files_skipped,execution_time_ms
,0,0,0,1,0


#### Conditional OPTIMIZE

In [0]:
file_count = (
    spark.sql("DESCRIBE DETAIL demo_raw_customers_delta_large")
    .select("numFiles")
    .collect()[0][0]
)

print("file_count : ", file_count)

if file_count > 1000:
    spark.sql("OPTIMIZE demo_raw_customers_delta_large")

file_count :  1


#### DEMONSTRATE OPTIMIZE impact

#### Demo: Force Small Files → OPTIMIZE → Observe Metrics

#### Step 1: Force small files (anti-pattern on purpose)

Why this works:
- `repartition(50)` creates many tiny output files
- Exactly the situation OPTIMIZE is designed to fix

In [0]:
# DEMO ONLY — intentionally create many small files
small_files_df = simulated_df.repartition(50)

(
    small_files_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("demo_raw_customers_delta_large")
)

#### Step 2: Run OPTIMIZE again

In [0]:
optimize_metrics = spark.sql(
    """
OPTIMIZE demo_raw_customers_delta_large
"""
)

#### Step 3: Inspect OPTIMIZE metrics (this time they matter)

In [0]:
from pyspark.sql import functions as F

optimize_metrics.select(
    "path",
    F.col("metrics.numFilesRemoved").alias("files_removed"),
    F.col("metrics.numFilesAdded").alias("files_added"),
    F.col("metrics.filesAdded.totalSize").alias("bytes_written"),
    F.col("metrics.totalFilesSkipped").alias("files_skipped"),
    F.col("metrics.totalTaskExecutionTimeMs").alias("execution_time_ms"),
).display()

path,files_removed,files_added,bytes_written,files_skipped,execution_time_ms
,2,1,1923,0,415
